# 動的計画法: 方策反復と価値反復(演習)

解説資料は `research-handbook/reinforcement-learning/dynamic-programming.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/dp_gridworld_solution.ipynb` にあるが、まず自力で取り組むこと。

**このnotebookで学ぶこと**

- 遷移確率テンソル `P[s,a,s']` と報酬 `R[s,a]` によるMDPの表現(モデル既知)
- 反復方策評価・方策反復・価値反復の実装
- 方策反復と価値反復の収束の速さの比較

## 環境: 4x4 gridworld

右下(状態15)がゴールの 4x4 格子世界です。ゴールに入るときだけ報酬 +1、それ以外は 0。壁にぶつかるとその場に留まります。動的計画法では環境モデル $p(s' \mid s, a)$ と $r(s,a)$ を**既知**として使える点が、後のQ学習との大きな違いです。

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

# 4x4 gridworld: 状態 0..15、右下 (状態15) がゴール
# 行動: 0=上, 1=下, 2=左, 3=右。壁にぶつかるとその場に留まる
N_ROWS, N_COLS = 4, 4
N_STATES = N_ROWS * N_COLS
N_ACTIONS = 4
GOAL = 15
GAMMA = 0.9

def build_mdp():
    """遷移確率テンソル P[s,a,s'] と報酬 R[s,a] を作る(モデル既知)"""
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))
    moves = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    for s in range(N_STATES):
        if s == GOAL:  # 終端状態: 自己ループ・報酬0
            P[s, :, s] = 1.0
            continue
        r, c = divmod(s, N_COLS)
        for a, (dr, dc) in moves.items():
            nr = min(max(r + dr, 0), N_ROWS - 1)
            nc = min(max(c + dc, 0), N_COLS - 1)
            ns = nr * N_COLS + nc
            P[s, a, ns] = 1.0
            if ns == GOAL:
                R[s, a] = 1.0  # ゴール到達で +1
    return P, R

P, R = build_mdp()
print("P shape:", P.shape, " R shape:", R.shape)
print("各 (s,a) で遷移確率の和は1:", np.allclose(P.sum(axis=2), 1.0))

## 課題1: 反復方策評価

ベルマン期待方程式をそのまま更新式にします:

$$V_{k+1}(s) = \sum_a \pi(a \mid s) \left[ r(s,a) + \gamma \sum_{s'} p(s' \mid s,a) V_k(s') \right]$$

numpyでは `Q = R + gamma * P @ V` の1行で全状態・全行動のバックアップが書けます。

In [ ]:
def policy_evaluation(pi, P, R, gamma=GAMMA, tol=1e-8):
    """反復方策評価: V^pi を求める。
    pi: (S,A) 行動選択確率, 戻り値: V (S,), スイープ回数"""
    V = np.zeros(N_STATES)
    for k in range(10000):
        # ---- ここから課題1 ----
        # ベルマン期待方程式で全状態を一括バックアップし、V_new (S,) を作る
        # ヒント: P @ V は (S,A) になる。pi(a|s) で重み付けして行動について和をとる
        Q = None
        V_new = None
        # ---- ここまで ----
        delta = np.max(np.abs(V_new - V))
        V = V_new
        if delta < tol:
            return V, k + 1
    return V, k + 1

# 一様ランダム方策を評価してみる
pi_uniform = np.ones((N_STATES, N_ACTIONS)) / N_ACTIONS
V_unif, n_sweeps = policy_evaluation(pi_uniform, P, R)
print(f"収束までのスイープ回数: {n_sweeps}")
print("V^pi (一様ランダム方策):")
print(V_unif.reshape(N_ROWS, N_COLS))

## 課題2: 方策反復

「評価 → 改善」を方策が変化しなくなるまで繰り返します。方策改善定理により、greedy化した方策は元の方策より悪くならないことが保証されています(`dynamic-programming.md` 参照)。

In [ ]:
def greedy_policy(V, P, R, gamma=GAMMA):
    """V に対する greedy 方策(方策改善)"""
    # ---- ここから課題2a ----
    # V から行動価値 Q (S,A) を計算する(課題1と同じバックアップ)
    Q = None
    # ---- ここまで ----
    pi = np.zeros((N_STATES, N_ACTIONS))
    best = np.argmax(Q, axis=1)
    pi[np.arange(N_STATES), best] = 1.0
    return pi

def policy_iteration(P, R, gamma=GAMMA):
    """方策反復 (課題2)。戻り値: 最適方策, V*, 改善ループ回数"""
    pi = np.ones((N_STATES, N_ACTIONS)) / N_ACTIONS
    for it in range(1000):
        # ---- ここから課題2 ----
        # (1) 現在の方策 pi を policy_evaluation で評価して V を得る
        # (2) greedy_policy で V に対する greedy 方策 pi_new を作る
        V = None
        pi_new = None
        # ---- ここまで ----
        if np.array_equal(pi_new, pi):              # 方策が変化しなければ最適
            return pi, V, it + 1
        pi = pi_new
    return pi, V, it + 1

pi_star, V_pi, n_pi = policy_iteration(P, R)
print(f"方策反復: {n_pi} 回の改善ループで収束")
print("V* (方策反復):")
print(V_pi.reshape(N_ROWS, N_COLS))
arrows = np.array(["↑", "↓", "←", "→"])
print("最適方策:")
print(arrows[np.argmax(pi_star, axis=1)].reshape(N_ROWS, N_COLS))

## 課題3: 価値反復

方策評価を収束まで待たず、ベルマン**最適**方程式で直接 $V^*$ に向かいます:

$$V_{k+1}(s) = \max_a \left[ r(s,a) + \gamma \sum_{s'} p(s' \mid s,a) V_k(s') \right]$$

方策反復と同じ $V^*$ に収束することを確認してください。

In [ ]:
def value_iteration(P, R, gamma=GAMMA, tol=1e-8):
    """価値反復 (課題3)。戻り値: V*, スイープ回数"""
    V = np.zeros(N_STATES)
    for k in range(10000):
        # ---- ここから課題3 ----
        # ベルマン最適方程式によるバックアップ: 期待値の代わりに max をとる
        V_new = None
        # ---- ここまで ----
        delta = np.max(np.abs(V_new - V))
        V = V_new
        if delta < tol:
            return V, k + 1
    return V, k + 1

V_vi, n_vi = value_iteration(P, R)
print(f"価値反復: {n_vi} スイープで収束")
print("V* (価値反復):")
print(V_vi.reshape(N_ROWS, N_COLS))
print("方策反復の V* と一致するか:", np.allclose(V_pi, V_vi, atol=1e-6))

## 発展課題

1. $\gamma$ を 0.5 / 0.9 / 0.99 に変えて、価値の伝わり方と収束の速さを観察する
2. 床を「滑る床」(確率0.2で横に滑る)にして最適価値がどう変わるか調べる

レポートには「結果」と「理由」を結びつけて書くこと(`bandit.md` 第7節と同じ作法)。

In [ ]:
# 発展課題1: 割引率 gamma を変えて最適価値関数を比較する
for g in [0.5, 0.9, 0.99]:
    V_g, k_g = value_iteration(P, R, gamma=g)
    print(f"gamma={g}: スイープ回数={k_g}, V[0]={V_g[0]:.4f}")
# gamma が大きいほど遠くのゴールの価値がスタート地点まで伝わる一方、収束は遅くなる

In [ ]:
# 発展課題2: 滑る床(意図した方向に 0.8、左右に 0.1 ずつ)にした場合
def build_slippery_mdp(p_slip=0.2):
    P0, _ = build_mdp()
    P = np.zeros_like(P0)
    R = np.zeros((N_STATES, N_ACTIONS))
    side = {0: (2, 3), 1: (2, 3), 2: (0, 1), 3: (0, 1)}  # 各行動の「横滑り」方向
    for s in range(N_STATES):
        if s == GOAL:
            P[s, :, s] = 1.0
            continue
        for a in range(N_ACTIONS):
            P[s, a] += (1 - p_slip) * P0[s, a]
            for b in side[a]:
                P[s, a] += (p_slip / 2) * P0[s, b]
            R[s, a] = P[s, a, GOAL] * 1.0  # ゴールに入る確率 × 報酬
    return P, R

P_s, R_s = build_slippery_mdp()
V_s, k_s = value_iteration(P_s, R_s)
print(f"滑る床: {k_s} スイープで収束")
print(V_s.reshape(N_ROWS, N_COLS))
print("決定的な床との差(滑ると価値が下がる):")
print((V_vi - V_s).reshape(N_ROWS, N_COLS))

## まとめ

- モデル既知なら、サンプリングせずにベルマン方程式の反復だけで最適方策が求まる
- 方策反復は少ない改善ループで収束するが、各ループ内で方策評価を回す
- 価値反復は方策評価を1スイープに省略した特殊ケースとみなせる
- 実際の問題ではモデルが未知 → サンプルから学ぶ **モンテカルロ法**(`monte-carlo.md`)・**TD学習**(`td-learning.md`)へ